# Study 944 — How Much Leverage — the teardown

`r_L = L·r_SPY − (L−1)·(^IRX + spread) − cost·turnover`, reset daily. The g(L) curve against Kelly `μ/σ²`, the Sharpe-invariance identity, a block bootstrap of the **argmax**, the rolling five-year hindsight optimum, the era hand-off, the **start-date sweep**, the ex-ante Kelly race tested on **log**-return differences, the two PROXY sweeps, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `d6dfd514b42d`, as-of 2026-06-30); the only live cells are explicitly synthetic.

In [1]:
R = {'start': '2003-06-04', 'end': '2026-06-30', 'n_days': 5799, 'fp': 'd6dfd514b42d', 'spread_bps': 50.0, 'cost_bps': 1.0, 'irx_ann': 1.47, 'bil_ann': 1.36, 'xcheck_gap_bps': 11, 'curve_lev': [1.0, 1.5, 2.0, 2.5, 2.85, 3.0], 'curve_tw': [11.68, 22.81, 36.35, 47.17, 49.99, 49.66], 'curve_cagr': [11.27, 14.56, 16.9, 18.23, 18.53, 18.49], 'curve_g': [10.68, 13.59, 15.61, 16.75, 17.0, 16.97], 'curve_sharpe': [0.575, 0.566, 0.56, 0.557, 0.555, 0.555], 'curve_vol': [18.6, 27.8, 37.1, 46.4, 52.9, 55.7], 'curve_dd': [-55.2, -72.6, -84.2, -91.3, -94.5, -95.5], 'curve_turn': [0.0, 1.4, 3.8, 7.2, 10.1, 11.5], 'opt': 2.85, 'kelly': 3.1, 'sharpe_gross_l1': 0.5752, 'sharpe_gross_l3': 0.5752, 'boot_ci_lo': 1.0, 'boot_ci_hi': 3.0, 'boot_sd': 0.6, 'boot_at_floor': 2.9, 'boot_at_cap': 44.4, 'boot_n': 1000, 'boot_block': 63, 'roll_n': 217, 'roll_mean': 2.36, 'roll_sd': 0.86, 'roll_min': 1.0, 'roll_max': 3.0, 'roll_at_floor': 24.0, 'roll_at_cap': 54.4, 'roll_kelly_min': -1.53, 'roll_kelly_max': 10.82, 'roll_years': [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026], 'roll_vals': [1.0, 1.0, 1.0, 1.0, 1.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 2.05, 2.85, 2.8, 3.0, 3.0], 'era_e_n': 2911, 'era_e_opt': 2.2, 'era_e_kelly': 2.37, 'era_e_cagr_opt': 11.93, 'era_e_cagr_l1': 8.82, 'era_e_dd_l1': -55.2, 'era_e_dd_l3': -95.5, 'era_l_n': 2888, 'era_l_opt': 3.0, 'era_l_kelly': 3.99, 'era_l_cagr_opt': 27.07, 'era_l_cagr_l1': 13.8, 'era_l_dd_l1': -33.7, 'era_l_dd_l3': -76.2, 'handoff_late_in_early': 10.56, 'handoff_early_unlev': 8.82, 'handoff_late_in_early_dd': -95.5, 'handoff_early_in_late': 23.52, 'handoff_late_opt_cagr': 27.07, 'ss_starts': ['2003-06-04', '2004-01-06', '2005-01-03', '2007-01-03', '2010-01-04'], 'ss_n': [5799, 5651, 5401, 4901, 4145], 'ss_opt': [2.85, 2.75, 2.65, 2.6, 3.0], 'ss_kelly': [3.1, 2.94, 2.88, 2.79, 4.55], 'ss_cagr_opt': [18.53, 17.03, 16.86, 16.8, 31.0], 'ss_handoff': [10.56, 7.07, 5.67, 2.66, 40.49], 'ss_unlev': [8.82, 7.81, 7.63, 6.98, 15.38], 'ss_edge': [1.73, -0.74, -1.96, -4.32, 25.11], 'kel_start': '2006-06-08', 'kel_end': '2026-06-30', 'kel_n': 5043, 'kel_mean_lev': 2.53, 'kel_at_cap': 64.0, 'kel_at_floor': 16.8, 'kel_tw': 33.95, 'kel_cagr': 19.26, 'kel_sharpe': 0.597, 'kel_vol': 41.2, 'kel_dd': -71.6, 'l1_tw': 8.61, 'l1_cagr': 11.36, 'l1_sharpe': 0.567, 'l1_vol': 19.4, 'l1_dd': -55.2, 'l2_tw': 22.35, 'l2_cagr': 16.79, 'l2_sharpe': 0.553, 'l2_dd': -84.2, 'adv_vs1': 6.86, 't_vs1': 1.2, 'ci_vs1_lo': -5.56, 'ci_vs1_hi': 17.52, 't_arith_vs1': 2.54, 'adv_vs2': 2.09, 't_vs2': 0.56, 'sh_diff': 0.03, 'sh_ci_lo': -0.18, 'sh_ci_hi': 0.254, 'sh_frac_pos': 59, 'caps': [1.5, 2.0, 2.5, 3.0], 'cap_adv': [2.78, 4.63, 5.91, 6.86], 'cap_t': [1.74, 1.46, 1.3, 1.2], 'cap_dd': [-60.3, -65.3, -68.9, -71.6], 'wins': [252, 504, 756, 1260], 'win_adv': [3.46, 4.95, 6.86, 5.12], 'win_t': [0.71, 0.93, 1.2, 0.78], 'spreads': [0, 25, 50, 100, 200], 'spread_opt': [3.0, 2.95, 2.85, 2.75, 2.45], 'spread_cagr': [19.71, 19.11, 18.53, 17.45, 15.56], 'spread_sharpe': [0.573, 0.564, 0.555, 0.539, 0.509], 'costs': [0.0, 1.0, 5.0], 'cost_opt': [2.9, 2.85, 2.8], 'cost_cagr': [18.66, 18.53, 18.06], 'syn_planted_mean': 1.91, 'syn_null_mean': 0.44, 'syn_planted_kelly': 2.04, 'syn_null_kelly': 0.01, 'syn_cond_argmax': 3.05, 'syn_cond_kelly': 3.04}

## 0. Provenance and the two assumptions

SPY total-return closes (`auto_adjust=True`) financed at ^IRX (13-week bill **discount** rate, act/360 on the previous close, accrued over the calendar days the bar spans). ^IRX rather than BIL because it covers the whole window; the two are cross-checked below. Two non-tape inputs, both **PROXIES**, both swept: the financing spread over bills and the one-way reset cost.

> 💡 **In plain words:** we are not measuring what it costs to borrow — we are *assuming* it, then checking how much the answer depends on the assumption.

In [2]:
print(f"window {R['start']} -> {R['end']}   n={R['n_days']:,}   fp={R['fp']}")
print(f"financing cross-check 2007-2026: ^IRX-implied {R['irx_ann']:.2f}%/yr vs "
      f"BIL total return {R['bil_ann']:.2f}%/yr  (gap {R['xcheck_gap_bps']:+d} bps/yr)")
print(f"PROXY spread {R['spread_bps']:.0f} bps/yr (swept 0-200) | "
      f"PROXY reset cost {R['cost_bps']:.0f} bp one-way (swept 0-5)")
print('one execution lag, used exactly once: the ex-ante Kelly estimate (through t, applied t+1)')

window 2003-06-04 -> 2026-06-30   n=5,799   fp=d6dfd514b42d
financing cross-check 2007-2026: ^IRX-implied 1.47%/yr vs BIL total return 1.36%/yr  (gap +11 bps/yr)
PROXY spread 50 bps/yr (swept 0-200) | PROXY reset cost 1 bp one-way (swept 0-5)
one execution lag, used exactly once: the ex-ante Kelly estimate (through t, applied t+1)


## 1. The g(L) curve, and the Sharpe identity that kills one axis

`g(L) ≈ L·μ − L²σ²/2 − (L−1)·s`, concave, peaking at `(μ−s)/σ²`. The excess return is `L·(r − r_f) − (L−1)·s − c`, so **gross of s and c the Sharpe is exactly invariant in L**. Reported below to four decimals as a construction check, not a finding.

In [3]:
print(f"{'L':>5}  {'terminal':>9}  {'CAGR':>8}  {'g(L)':>8}  {'exSharpe':>9}  "
      f"{'vol':>7}  {'maxDD':>8}  {'turn/yr':>8}")
for i, L in enumerate(R['curve_lev']):
    mark = '  <-- realised optimum' if L == R['opt'] else ''
    print(f"{L:5.2f}  {R['curve_tw'][i]:8.2f}x  {R['curve_cagr'][i]:+7.2f}%  "
          f"{R['curve_g'][i]:+7.2f}%  {R['curve_sharpe'][i]:+9.3f}  "
          f"{R['curve_vol'][i]:6.1f}%  {R['curve_dd'][i]:+7.1f}%  "
          f"{R['curve_turn'][i]:7.1f}x{mark}")
print(f"\nrealised argmax {R['opt']:.2f}   theoretical Kelly mu/sigma^2 {R['kelly']:.2f}")
print(f"Sharpe invariance (0 bps spread, 0 cost): L=1 {R['sharpe_gross_l1']:.4f}  "
      f"L=3 {R['sharpe_gross_l3']:.4f}  -> identical by construction")

    L   terminal      CAGR      g(L)   exSharpe      vol     maxDD   turn/yr
 1.00     11.68x   +11.27%   +10.68%     +0.575    18.6%    -55.2%      0.0x
 1.50     22.81x   +14.56%   +13.59%     +0.566    27.8%    -72.6%      1.4x
 2.00     36.35x   +16.90%   +15.61%     +0.560    37.1%    -84.2%      3.8x
 2.50     47.17x   +18.23%   +16.75%     +0.557    46.4%    -91.3%      7.2x
 2.85     49.99x   +18.53%   +17.00%     +0.555    52.9%    -94.5%     10.1x  <-- realised optimum
 3.00     49.66x   +18.49%   +16.97%     +0.555    55.7%    -95.5%     11.5x

realised argmax 2.85   theoretical Kelly mu/sigma^2 3.10
Sharpe invariance (0 bps spread, 0 cost): L=1 0.5752  L=3 0.5752  -> identical by construction


The realised peak sits **below** Kelly (2.85 vs 3.10), as it must: the closed form assumes Gaussian returns and free financing, and both the fat left tail and the 50 bps spread push the optimum down. Directionally correct machinery — which is the only thing this comparison is allowed to establish.

## 2. Is the argmax identified? Block bootstrap

Circular block bootstrap, 1000 draws, 63-day blocks of the joint `(r_asset, r_cash)` rows so vol clustering survives; the **whole grid is re-solved** on each resample and the argmax recorded.

> 💡 **In plain words:** we shuffle history in quarterly chunks and ask, a thousand times, 'what leverage would have been best?' If the answer were knowable, the thousand answers would cluster.

In [4]:
print(f"argmax {R['opt']:.2f}   95% CI [{R['boot_ci_lo']:.2f}, {R['boot_ci_hi']:.2f}]  "
      f"sd {R['boot_sd']:.2f}")
print(f"draws at the 1.00 floor: {R['boot_at_floor']:.1f}%   "
      f"draws at the 3.00 cap: {R['boot_at_cap']:.1f}%")
print('-> the CI is the entire grid: the location of the peak is not identified')

argmax 2.85   95% CI [1.00, 3.00]  sd 0.60
draws at the 1.00 floor: 2.9%   draws at the 3.00 cap: 44.4%
-> the CI is the entire grid: the location of the peak is not identified


This is not estimator weakness, it is the arithmetic of `L* = μ/σ²`. With σ ≈ 19%/yr and 23.1 years, `se(μ) ≈ 3.9%/yr`, so `se(L*) ≈ 1.1` — Merton (1980) in one line. Variance converges with sampling frequency; the mean only with calendar span.

## 3. Is the argmax stable? Rolling five-year hindsight optimum

217 windows of 1,260 days, monthly stride, each solved with **perfect hindsight inside the window** — the easiest possible version of the problem.

In [5]:
print(f"{R['roll_n']} windows: mean {R['roll_mean']:.2f}  sd {R['roll_sd']:.2f}  "
      f"range [{R['roll_min']:.2f}, {R['roll_max']:.2f}]")
print(f"at the 1.00 floor {R['roll_at_floor']:.1f}% of windows  |  "
      f"at the 3.00 cap {R['roll_at_cap']:.1f}%")
print(f"rolling Kelly estimate ranges [{R['roll_kelly_min']:+.2f}, {R['roll_kelly_max']:+.2f}]")
print('\nyear-end reading:')
for y, v in zip(R['roll_years'], R['roll_vals']):
    print(f'  {y}  {v:4.2f}x  ' + '#' * int(round(v * 12)))

217 windows: mean 2.36  sd 0.86  range [1.00, 3.00]
at the 1.00 floor 24.0% of windows  |  at the 3.00 cap 54.4%
rolling Kelly estimate ranges [-1.53, +10.82]

year-end reading:
  2008  1.00x  ############
  2009  1.00x  ############
  2010  1.00x  ############
  2011  1.00x  ############
  2012  1.00x  ############
  2013  3.00x  ####################################
  2014  3.00x  ####################################
  2015  3.00x  ####################################
  2016  3.00x  ####################################
  2017  3.00x  ####################################
  2018  3.00x  ####################################
  2019  3.00x  ####################################
  2020  3.00x  ####################################
  2021  3.00x  ####################################
  2022  2.05x  #########################
  2023  2.85x  ##################################
  2024  2.80x  ##################################
  2025  3.00x  ####################################
  2026  3.00x  ######

## 4. The era cut, and the hand-off test

Split 2015-01-01. A stable optimum should survive the split; the hand-off asks the practical question directly — take one era's answer, use it in the other.

In [6]:
print(f"early n={R['era_e_n']}: optimum {R['era_e_opt']:.2f}  Kelly {R['era_e_kelly']:.2f}  "
      f"CAGR@opt {R['era_e_cagr_opt']:+.2f}%  CAGR@1 {R['era_e_cagr_l1']:+.2f}%  "
      f"DD@1 {R['era_e_dd_l1']:+.1f}%  DD@3 {R['era_e_dd_l3']:+.1f}%")
print(f"late  n={R['era_l_n']}: optimum {R['era_l_opt']:.2f}  Kelly {R['era_l_kelly']:.2f}  "
      f"CAGR@opt {R['era_l_cagr_opt']:+.2f}%  CAGR@1 {R['era_l_cagr_l1']:+.2f}%  "
      f"DD@1 {R['era_l_dd_l1']:+.1f}%  DD@3 {R['era_l_dd_l3']:+.1f}%")
print()
print(f"hand-off: late optimum {R['era_l_opt']:.2f}x applied 2004-2014 -> "
      f"{R['handoff_late_in_early']:+.2f}%/yr vs {R['handoff_early_unlev']:+.2f}%/yr unlevered "
      f"(DD {R['handoff_late_in_early_dd']:+.1f}%)")
print(f"          early optimum {R['era_e_opt']:.2f}x applied 2015-2026 -> "
      f"{R['handoff_early_in_late']:+.2f}%/yr vs {R['handoff_late_opt_cagr']:+.2f}%/yr at the late optimum")

early n=2911: optimum 2.20  Kelly 2.37  CAGR@opt +11.93%  CAGR@1 +8.82%  DD@1 -55.2%  DD@3 -95.5%
late  n=2888: optimum 3.00  Kelly 3.99  CAGR@opt +27.07%  CAGR@1 +13.80%  DD@1 -33.7%  DD@3 -76.2%

hand-off: late optimum 3.00x applied 2004-2014 -> +10.56%/yr vs +8.82%/yr unlevered (DD -95.5%)
          early optimum 2.20x applied 2015-2026 -> +23.52%/yr vs +27.07%/yr at the late optimum


### 4b. The start-date sweep — the sharpest instability in the study

The era hand-off above depends on a boundary the analyst picks. So does the *sample start*, and that one is usually invisible: it is set by whatever history the data vendor happens to hold. Re-run the entire headline from five different start dates, holding the split, the as-of date and every parameter fixed.

> ⚠️ **Why this section exists.** The first draft of this study ran on a cache whose ^IRX began 2004-01-06 and reported the hand-off with the **opposite sign** (+7.07% vs +7.81% unlevered — a loss). A later cache refresh pushed the start back to 2003-06-04 and the conclusion flipped. Nothing about the world changed. Rather than silently restate the headline, the sensitivity is now measured and shipped.

In [7]:
print(f"{'start':>12}{'n':>7}{'opt':>7}{'Kelly':>8}{'CAGR@opt':>10}"
      f"{'handoff':>10}{'unlev':>8}{'edge':>9}")
for i, s0 in enumerate(R['ss_starts']):
    print(f"{s0:>12}{R['ss_n'][i]:7d}{R['ss_opt'][i]:7.2f}{R['ss_kelly'][i]:8.2f}"
          f"{R['ss_cagr_opt'][i]:+9.2f}%{R['ss_handoff'][i]:+9.2f}%"
          f"{R['ss_unlev'][i]:+7.2f}%{R['ss_edge'][i]:+8.2f}%")
print()
print(f"realised optimum ranges {min(R['ss_opt']):.2f} -> {max(R['ss_opt']):.2f} "
      f"and the hand-off edge {min(R['ss_edge']):+.2f}%/yr -> {max(R['ss_edge']):+.2f}%/yr")
print('-> the sign of the study\'s central claim is a function of the left edge of the window')

       start      n    opt   Kelly  CAGR@opt   handoff   unlev     edge
  2003-06-04   5799   2.85    3.10   +18.53%   +10.56%  +8.82%   +1.73%
  2004-01-06   5651   2.75    2.94   +17.03%    +7.07%  +7.81%   -0.74%
  2005-01-03   5401   2.65    2.88   +16.86%    +5.67%  +7.63%   -1.96%
  2007-01-03   4901   2.60    2.79   +16.80%    +2.66%  +6.98%   -4.32%
  2010-01-04   4145   3.00    4.55   +31.00%   +40.49% +15.38%  +25.11%

realised optimum ranges 2.60 -> 3.00 and the hand-off edge -4.32%/yr -> +25.11%/yr
-> the sign of the study's central claim is a function of the left edge of the window


Put the spread sweep next to this. Moving the financing assumption across its entire 0-200 bps range moves the optimum by 0.55. Moving the *start date* by a few years moves it by 0.40, and moves the hand-off conclusion from +1.73%/yr to -4.32%/yr (and to +25.11%/yr if you start after the crash). **The arbitrary choice nobody documents dominates the assumption everybody argues about.**

## 5. The tradable arm — ex-ante Kelly, one lag, log-growth test

`μ/σ²` on the trailing 756 days through *t*, clipped to [1, 3], applied at *t+1*. Raced against fixed multiples over the same window.

The test statistic is the HAC *t* on the daily **log**-return difference, because terminal wealth is a product and growth is additive in logs. The arithmetic-excess *t* is printed alongside precisely so the divergence is visible: it is mechanically inflated by the higher-vol arm and is **not** the test a compounding claim must pass.

> 💡 **In plain words:** a levered arm has a bigger *average* daily return almost by definition. Only the log difference answers 'did it actually end up with more money, reliably?'

In [8]:
print(f"window {R['kel_start']} -> {R['kel_end']}  n={R['kel_n']:,}")
print(f"applied multiple: mean {R['kel_mean_lev']:.2f}, at cap {R['kel_at_cap']:.1f}% "
      f"of days, at floor {R['kel_at_floor']:.1f}%")
print()
print(f"{'arm':<16}{'terminal':>10}{'CAGR':>9}{'exSharpe':>10}{'vol':>8}{'maxDD':>9}")
print(f"{'ex-ante Kelly':<16}{R['kel_tw']:9.2f}x{R['kel_cagr']:+8.2f}%"
      f"{R['kel_sharpe']:+10.3f}{R['kel_vol']:7.1f}%{R['kel_dd']:+8.1f}%")
print(f"{'fixed L=1.00':<16}{R['l1_tw']:9.2f}x{R['l1_cagr']:+8.2f}%"
      f"{R['l1_sharpe']:+10.3f}{R['l1_vol']:7.1f}%{R['l1_dd']:+8.1f}%")
print(f"{'fixed L=2.00':<16}{R['l2_tw']:9.2f}x{R['l2_cagr']:+8.2f}%"
      f"{R['l2_sharpe']:+10.3f}{'':7}{R['l2_dd']:+8.1f}%")
print()
print(f"vs L=1: log-growth advantage {R['adv_vs1']:+.2f}%/yr  HAC t={R['t_vs1']:+.2f}  "
      f"95% CI [{R['ci_vs1_lo']:+.2f}%, {R['ci_vs1_hi']:+.2f}%]  "
      f"(arithmetic-excess t={R['t_arith_vs1']:+.2f} <- inflated, not the test)")
print(f"vs L=2: log-growth advantage {R['adv_vs2']:+.2f}%/yr  HAC t={R['t_vs2']:+.2f}")

window 2006-06-08 -> 2026-06-30  n=5,043
applied multiple: mean 2.53, at cap 64.0% of days, at floor 16.8%

arm               terminal     CAGR  exSharpe     vol    maxDD
ex-ante Kelly       33.95x  +19.26%    +0.597   41.2%   -71.6%
fixed L=1.00         8.61x  +11.36%    +0.567   19.4%   -55.2%
fixed L=2.00        22.35x  +16.79%    +0.553          -84.2%

vs L=1: log-growth advantage +6.86%/yr  HAC t=+1.20  95% CI [-5.56%, +17.52%]  (arithmetic-excess t=+2.54 <- inflated, not the test)
vs L=2: log-growth advantage +2.09%/yr  HAC t=+0.56


### 5a. The one place a multiple *can* move Sharpe

Constant leverage cannot touch Sharpe — that is the identity above. But this arm is *time-varying*, so it can, and it does: +0.030. That is exactly the number a reader would seize on, so it gets a paired block-bootstrap CI rather than a shrug.

In [9]:
print(f"ex-ante Kelly {R['kel_sharpe']:+.3f}  vs  fixed 1x {R['l1_sharpe']:+.3f}")
print(f"difference {R['sh_diff']:+.3f}   95% CI "
      f"[{R['sh_ci_lo']:+.3f}, {R['sh_ci_hi']:+.3f}]   "
      f"{R['sh_frac_pos']}% of draws positive")
print('-> the CI is ~14x the size of the point estimate: not distinguishable from zero')

ex-ante Kelly +0.597  vs  fixed 1x +0.567
difference +0.030   95% CI [-0.180, +0.254]   59% of draws positive
-> the CI is ~14x the size of the point estimate: not distinguishable from zero


### 5b. The advantage is a knob, not a signal

Raise the cap and the advantage rises while the *t*-stat **falls**. A genuine signal scales its *t* with its magnitude; a leverage knob does not. Same story for the estimation window. No configuration reaches |*t*| = 2.

In [10]:
print('cap sweep:')
for c, a, t, dd in zip(R['caps'], R['cap_adv'], R['cap_t'], R['cap_dd']):
    print(f'  cap {c:.1f}: advantage {a:+.2f}%/yr  t={t:+.2f}  maxDD {dd:+.1f}%')
print('\nestimation-window sweep:')
for w, a, t in zip(R['wins'], R['win_adv'], R['win_t']):
    print(f'  {w:5d}d: advantage {a:+.2f}%/yr  t={t:+.2f}')
print(f"\nbest |t| anywhere in the design space: {max(R['cap_t'] + R['win_t']):.2f}  "
      f"(desk bar for Real: |t| >= 2)")

cap sweep:
  cap 1.5: advantage +2.78%/yr  t=+1.74  maxDD -60.3%
  cap 2.0: advantage +4.63%/yr  t=+1.46  maxDD -65.3%
  cap 2.5: advantage +5.91%/yr  t=+1.30  maxDD -68.9%
  cap 3.0: advantage +6.86%/yr  t=+1.20  maxDD -71.6%

estimation-window sweep:
    252d: advantage +3.46%/yr  t=+0.71
    504d: advantage +4.95%/yr  t=+0.93
    756d: advantage +6.86%/yr  t=+1.20
   1260d: advantage +5.12%/yr  t=+0.78

best |t| anywhere in the design space: 1.74  (desk bar for Real: |t| >= 2)


## 6. PROXY sweeps — testing the assumptions rather than trusting them

In [11]:
print('financing spread over bills (bps/yr):')
for s, o, c, sh in zip(R['spreads'], R['spread_opt'], R['spread_cagr'], R['spread_sharpe']):
    print(f'  {s:4d}: optimum {o:.2f}x  CAGR@opt {c:+.2f}%  exSharpe@opt {sh:+.3f} '
          f"(unlevered {R['curve_sharpe'][0]:+.3f})")
print('\none-way reset cost (bps):')
for c, o, g in zip(R['costs'], R['cost_opt'], R['cost_cagr']):
    print(f'  {c:4.1f}: optimum {o:.2f}x  CAGR@opt {g:+.2f}%')
print('\n-> the borrow bites, the reset does not; and at EVERY assumption the')
print('   excess Sharpe at the optimum sits below the unlevered Sharpe.')

financing spread over bills (bps/yr):


     0: optimum 3.00x  CAGR@opt +19.71%  exSharpe@opt +0.573 (unlevered +0.575)
    25: optimum 2.95x  CAGR@opt +19.11%  exSharpe@opt +0.564 (unlevered +0.575)
    50: optimum 2.85x  CAGR@opt +18.53%  exSharpe@opt +0.555 (unlevered +0.575)
   100: optimum 2.75x  CAGR@opt +17.45%  exSharpe@opt +0.539 (unlevered +0.575)
   200: optimum 2.45x  CAGR@opt +15.56%  exSharpe@opt +0.509 (unlevered +0.575)

one-way reset cost (bps):
   0.0: optimum 2.90x  CAGR@opt +18.66%
   1.0: optimum 2.85x  CAGR@opt +18.53%
   5.0: optimum 2.80x  CAGR@opt +18.06%

-> the borrow bites, the reset does not; and at EVERY assumption the
   excess Sharpe at the optimum sits below the unlevered Sharpe.


## 7. Live synthetic control — the machinery is unbiased

**Synthetic, not the real tape.** An i.i.d. Student-*t* world with a *planted* growth-optimal leverage of 2.0, and a null version earning exactly cash. The sweep must recover 2.0 on the first and collapse to the floor on the second. Conditional consistency (argmax vs the tape's own in-sample `μ/σ²`) is checked on a single tape, where it is near-exact.

The per-seed scatter is not noise to be apologised for — it is the study's result, reproduced in a laboratory where the DGP is *known and stationary*.

In [12]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from optimal_leverage import data, strategy as st
GRID = np.round(np.arange(0.0, 3.0001, 0.25), 4)
for tag, ss in [('planted Kelly = 2.0', 1.0), ('null (zero excess drift)', 0.0)]:
    opts, kels = [], []
    for s in range(8):
        d = st.synthetic_detect(data.synthetic_daily(signal_strength=ss, seed=944+s)[0], grid=GRID)
        opts.append(d['opt_lev']); kels.append(d['kelly'])
    print(f"{tag:26s} argmax mean {np.mean(opts):.2f} (sd {np.std(opts, ddof=1):.2f})  "
          f"Kelly mean {np.mean(kels):+.2f}")
    print(f"{'':26s} per-seed argmax {sorted(opts)}")
lg = st.prepare_synth(data.synthetic_daily(signal_strength=1.0, seed=944)[0])
fine = st.realised_optimum(lg, grid=np.round(np.arange(0.0, 4.0001, 0.05), 4),
                           spread_bps=0.0, cost_bps=0.0)
print(f"\nconditional consistency on one 40-year tape: argmax {fine:.2f} vs "
      f"in-sample Kelly {st.kelly_from_legs(lg):.2f}")

planted Kelly = 2.0        argmax mean 1.91 (sd 0.89)  Kelly mean +2.04
                           per-seed argmax [1.0, 1.0, 1.0, 1.5, 2.25, 2.5, 3.0, 3.0]


null (zero excess drift)   argmax mean 0.44 (sd 0.64)  Kelly mean +0.01
                           per-seed argmax [0.0, 0.0, 0.0, 0.0, 0.25, 0.5, 1.0, 1.75]



conditional consistency on one 40-year tape: argmax 3.05 vs in-sample Kelly 3.04


## Verdict

- **Signal — Weak.** The growth curve is real, concave and peaks at **2.85×** against a Kelly of 3.10× — directionally exactly what theory predicts, and positive in both eras and at every assumption. But the *location* of the peak is unidentified: block-bootstrap CI **[1.00, 3.00]** (the whole grid, sd 0.60); the five-year hindsight optimum sits at the floor in 24% of windows and the cap in 54%; the ex-ante Kelly arm beats 1× by +6.86%/yr at HAC *t* = +1.20 with a CI spanning zero, and by only +2.09%/yr (*t* = +0.56) against an arbitrary fixed 2×. Best |*t*| anywhere in the design space: 1.74. Positive sign everywhere, significance nowhere.
- **Tradability — Mirage.** Excess Sharpe is invariant in L by construction (0.5752 at 1× and 3× gross of financing) and falls to 0.555 at the optimum once the spread is paid — nothing risk-adjusted exists to bank. The growth is bought with a -94.5% drawdown, and the number you would have to know is not merely noisy but sample-dependent: the identical hand-off test reads +1.73%/yr or -4.32%/yr against not levering at all depending only on where the window starts. The late optimum applied in the early era did stay ahead (+10.56%/yr vs +8.82%) — through a -95.5% drawdown.
- **Caveat, named and priced.** The window is gated by cached ^IRX history and opens in 2003-06: it contains the GFC and 2022 but not 2000-2002. A sample opening in 2000 would place the realised optimum lower still — section 4b measures how much that boundary is worth rather than merely confessing it. The whole result is one realisation of one index that did not suffer a terminal decade — survivorship in its macro form, and it belongs on the Signal axis.